# In-memory retrieval and RAG (Chapter 8)

This notebook builds the chapter's `dspy.retrievers.Embeddings` index, uses it in a fixed RAG module, and exposes the same retriever as a just-in-time ReAct tool.

**Required environment variable:** `OPENAI_API_KEY`.


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


## Build and query an in-memory semantic index


In [ ]:
corpus = [
    "DSPy is a framework for programming language models.",
    "Signatures define input/output specifications for DSPy tasks.",
    "Optimizers like BootstrapFewShot automatically improve prompts.",
    "ReAct agents use tools to gather information iteratively.",
]

embedder = dspy.Embedder(
    "openai/text-embedding-3-small",
    dimensions=512,
)
search = dspy.retrievers.Embeddings(
    embedder=embedder,
    corpus=corpus,
    k=3,
)

results = search("What are DSPy optimizers?")
print(results.passages)


## Fixed RAG pipeline


In [ ]:
class SimpleRAG(dspy.Module):
    def __init__(self, retriever):
        super().__init__()
        self.retriever = retriever
        self.respond = dspy.ChainOfThought(
            "context, question -> response"
        )

    def forward(self, question):
        context = self.retriever(question).passages
        return self.respond(
            context=context,
            question=question,
        )


rag = SimpleRAG(retriever=search)
rag_result = rag(question="How do I optimize a DSPy program?")
print(rag_result.response)


## Agent-controlled retrieval

The same retriever becomes a tool when retrieval should happen only if—and as many times as—the agent decides it is useful.


In [ ]:
def search_knowledge_base(query: str) -> str:
    """Search the internal knowledge base for relevant passages."""
    passages = search(query).passages
    return "\n\n".join(passages)


rag_agent = dspy.ReAct(
    "question -> answer",
    tools=[search_knowledge_base],
    max_iters=5,
)

agent_result = rag_agent(
    question="How do I optimize a DSPy program?"
)
print(agent_result.answer)
